# Generating mock datasets from a GP with a given covariance kernel

This notebook demonstrates how to use `pgmuvi` to **generate synthetic (mock) light curves
by drawing samples from a Gaussian Process prior** with a chosen covariance kernel.  
This is useful for:

* Validating that your fitted GP model really does reproduce the observed light-curve
  properties (e.g. period, coherence length).
* Testing analysis pipelines before observing a new source.
* Producing realistic realisations of different variability types for comparison studies.

## Two complementary approaches

| Approach | What it gives you | When to use it |
|---|---|---|
| **`pgmuvi.synthetic`** | Deterministic signal (sinusoid) plus noise | Quick tests; known, simple shapes |
| **GP prior sample** | Realisations drawn from a proper GP prior | Scientifically faithful mock data matching the full kernel structure |

In [ ]:
# Install pgmuvi if running in a fresh environment
try:
    import pgmuvi
except ModuleNotFoundError:
    %pip install git+https://github.com/ICSM/pgmuvi.git
    import pgmuvi

In [ ]:
import torch
import gpytorch
import numpy as np
import matplotlib.pyplot as plt

# Fix random seeds for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

---
## 1. Quick approach: `pgmuvi.synthetic`

`pgmuvi.synthetic` provides convenience functions that generate a `Lightcurve` directly
from a *deterministic* signal (a sinusoid) plus optional noise — no GP machinery needed.

Use this when you just want a realistic-looking periodic light curve to test your pipeline.

In [ ]:
from pgmuvi.synthetic import make_simple_sinusoid_1d, make_multi_sinusoid_1d
import math

# --- Single sinusoid --------------------------------------------------------
lc_sin = make_simple_sinusoid_1d(
    n_obs=120,
    period=150.0,       # dominant period in days
    amplitude=1.0,
    noise_level=0.08,   # fractional Poisson-like noise
    noise_type="poisson",
    t_span=600.0,       # total baseline in days
    irregular=True,     # irregular cadence (more realistic)
    seed=SEED,
)

# --- Multi-component sinusoid -----------------------------------------------
# make_multi_sinusoid_1d takes a list of component dicts; each component
# requires 'period', 'amplitude', and 'phase' keys.
lc_multi = make_multi_sinusoid_1d(
    n_obs=150,
    components=[
        {"period": 200.0, "amplitude": 1.0, "phase": 0.0},
        {"period": 100.0, "amplitude": 0.4, "phase": math.pi / 4},
        {"period": 50.0,  "amplitude": 0.2, "phase": math.pi / 2},
    ],
    noise_level=0.05,
    seed=SEED,
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, lc, title in zip(
    axes,
    [lc_sin, lc_multi],
    ["Simple sinusoid (P = 150 d)", "Multi-component sinusoid (P ≈ 200 d)"],
):
    t_np = lc.xdata.numpy()
    y_np = lc.ydata.numpy()
    yerr_np = lc.yerr.numpy() if lc.yerr is not None else None
    if yerr_np is not None:
        ax.errorbar(t_np, y_np, yerr=yerr_np, fmt="k.", elinewidth=0.8, alpha=0.7)
    else:
        ax.plot(t_np, y_np, "k.")
    ax.set_title(title)
    ax.set_xlabel("Time (days)")
    ax.set_ylabel("Flux")
plt.tight_layout()
plt.show()

The returned object is already a `Lightcurve` that you can pass to `lc.fit()` or any
other `pgmuvi` analysis method without further wrapping.

---
## 2. Sampling from a GP prior

A more powerful approach is to draw realisations directly from the **GP prior distribution**
defined by your chosen kernel.  This guarantees that the mock data have exactly the
covariance structure you specify.

The recipe is:

1. Choose a grid of observation times.
2. Instantiate a `pgmuvi` GP model with the desired kernel.
3. Set the kernel hyperparameters to the desired values.
4. Call `model.forward(t)` in evaluation mode to obtain the prior distribution at those
   times.
5. Call `.sample()` on the distribution.
6. Optionally add independent observational noise.
7. Wrap the result in a `Lightcurve`.

We demonstrate this for three physically motivated kernels that `pgmuvi` supports.

### 2a. Quasi-periodic kernel

The quasi-periodic kernel is the product of a `PeriodicKernel` and an `RBFKernel`:

$$k_{\rm QP}(t, t') = \sigma^2 \; k_{\rm periodic}(t, t') \; k_{\rm RBF}(t, t')$$

This captures signals that are *approximately* periodic but whose amplitude and phase
drift slowly — a good description of many long-period variable (LPV) stars.

Key hyperparameters:

| Parameter | What it controls |
|---|---|
| `period_length` (PeriodicKernel) | Dominant oscillation period |
| `lengthscale` (RBFKernel) | Coherence timescale — how fast the period/amplitude drifts |
| `outputscale` (ScaleKernel) | Overall amplitude variance |

In [ ]:
from pgmuvi.gps import QuasiPeriodicGPModel
from pgmuvi.lightcurve import Lightcurve

torch.manual_seed(SEED)

# 1. Observation times: 150 irregular points over 600 days
rng = np.random.default_rng(SEED)
t_np = np.sort(rng.uniform(0, 600, 150))
t = torch.as_tensor(t_np, dtype=torch.float32)

# 2. Instantiate the GP model (training data are placeholders — we only need
#    them to satisfy the GPyTorch ExactGP constructor)
likelihood_qp = gpytorch.likelihoods.GaussianLikelihood()
model_qp = QuasiPeriodicGPModel(
    t, torch.zeros_like(t), likelihood_qp, period=100.0
)

# 3. Set the kernel hyperparameters
TRUE_PERIOD = 100.0          # days
TRUE_COHERENCE = 800.0       # days — very coherent (slow drift)
TRUE_AMPLITUDE_VAR = 1.5     # output variance (σ²)

# The quasi-periodic kernel is ScaleKernel(ProductKernel(Periodic, RBF))
qp_kernel = model_qp.covar_module  # ScaleKernel
qp_kernel.outputscale = TRUE_AMPLITUDE_VAR
qp_kernel.base_kernel.kernels[0].period_length = TRUE_PERIOD     # PeriodicKernel
qp_kernel.base_kernel.kernels[1].lengthscale = TRUE_COHERENCE    # RBFKernel

# 4 & 5. Draw a sample from the prior
model_qp.eval()
likelihood_qp.eval()

with torch.no_grad():
    prior_dist = model_qp.forward(t)
    f_true = prior_dist.sample()          # noise-free GP realisation

# 6. Add observational noise
NOISE_SIGMA = 0.12
y_obs = f_true + NOISE_SIGMA * torch.randn_like(f_true)
yerr = torch.full_like(f_true, NOISE_SIGMA)

# 7. Wrap in a Lightcurve
lc_qp = Lightcurve(t, y_obs, yerr=yerr, max_samples=None)

print(
    f"Lightcurve created: {len(lc_qp.xdata)} observations over"
    f" {float(lc_qp.xdata.max() - lc_qp.xdata.min()):.0f} days"
)

# --- Plot -------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(
    t_np,
    f_true.numpy(),
    "-",
    color="steelblue",
    lw=1.5,
    label=f"GP prior sample (P = {TRUE_PERIOD} d)",
)
ax.errorbar(
    t_np,
    y_obs.numpy(),
    yerr=NOISE_SIGMA,
    fmt="k.",
    elinewidth=0.7,
    alpha=0.6,
    label="Noisy observations",
)
ax.set_xlabel("Time (days)")
ax.set_ylabel("Flux")
ax.set_title("Mock light curve from quasi-periodic GP prior")
ax.legend()
plt.tight_layout()
plt.show()

### 2b. Matérn kernel (stochastic / red-noise variability)

The Matérn kernel produces smooth stochastic (aperiodic) processes commonly used to
model AGN accretion-disk variability or red-noise stellar activity.

| Parameter | What it controls |
|---|---|
| `nu` | Smoothness of the process (0.5 = Ornstein-Uhlenbeck; 1.5 or 2.5 = smoother) |
| `lengthscale` | Characteristic variability timescale |
| `outputscale` | Overall amplitude variance |

In [ ]:
from pgmuvi.gps import MaternGPModel

torch.manual_seed(SEED + 1)

# Dense, regularly spaced grid to show the smooth process clearly
t_mat = torch.linspace(0, 500, 200)

likelihood_mat = gpytorch.likelihoods.GaussianLikelihood()
model_mat = MaternGPModel(
    t_mat,
    torch.zeros_like(t_mat),
    likelihood_mat,
    nu=1.5,
    lengthscale=80.0,   # ~80-day variability timescale
)
model_mat.covar_module.outputscale = 2.0   # amplitude variance σ²

model_mat.eval()
likelihood_mat.eval()

with torch.no_grad():
    prior_mat = model_mat.forward(t_mat)
    # Draw 3 independent realisations to visualise the prior distribution
    samples_mat = prior_mat.sample(torch.Size([3]))

fig, ax = plt.subplots(figsize=(10, 4))
colors = ["steelblue", "darkorange", "seagreen"]
for i, (s, c) in enumerate(zip(samples_mat, colors)):
    ax.plot(
        t_mat.numpy(),
        s.numpy(),
        color=c,
        lw=1.5,
        alpha=0.85,
        label=f"Realisation {i + 1}",
    )
ax.set_xlabel("Time (days)")
ax.set_ylabel("Flux")
ax.set_title(
    r"Three realisations from a Matérn-3/2 GP prior"
    " (lengthscale = 80 d, σ² = 2)"
)
ax.legend()
plt.tight_layout()
plt.show()

### 2c. Spectral Mixture kernel (pgmuvi's default)

The Spectral Mixture Kernel (SMK) is the default kernel in `pgmuvi`.  It models the
Power Spectral Density (PSD) of the covariance as a sum of Gaussian components,
making it flexible enough to represent virtually any stationary covariance structure.

Each mixture component is characterised by three parameters:

| Parameter | Shape | What it controls |
|---|---|---|
| `mixture_means` | `(Q, 1, 1)` | Centre frequency of each component (= 1/period) |
| `mixture_scales` | `(Q, 1, 1)` | Bandwidth of each Gaussian in frequency space (narrow = long coherence) |
| `mixture_weights` | `(Q,)` | Relative amplitude of each component |

where `Q` is the number of mixture components.

**Tip:** `mixture_means` are in cycles per (time unit), so for a period of *P* days set
`mixture_means = 1/P`.

In [ ]:
from pgmuvi.gps import SpectralMixtureGPModel

torch.manual_seed(SEED + 2)

# Observation times: 180 points over 800 days
rng2 = np.random.default_rng(SEED + 2)
t_sm_np = np.sort(rng2.uniform(0, 800, 180))
t_sm = torch.as_tensor(t_sm_np, dtype=torch.float32)

# Q = 2 mixture components
Q = 2
likelihood_sm = gpytorch.likelihoods.GaussianLikelihood()
model_sm = SpectralMixtureGPModel(
    t_sm, torch.zeros_like(t_sm), likelihood_sm, num_mixtures=Q
)

# Define two spectral components:
#   Component 1: dominant period ~ 200 days, moderately coherent
#   Component 2: secondary period ~ 60 days, less coherent (broader bandwidth)
P1, P2 = 200.0, 60.0         # periods in days
BW1, BW2 = 5e-4, 3e-3        # bandwidths in cycles/day (narrow = more coherent)
W1, W2 = 0.65, 0.35          # relative weights

model_sm.covar_module.initialize(
    **{
        "mixture_means": torch.tensor([[[1.0 / P1]], [[1.0 / P2]]]),
        "mixture_scales": torch.tensor([[[BW1]], [[BW2]]]),
        "mixture_weights": torch.tensor([W1, W2]),
    }
)

model_sm.eval()
likelihood_sm.eval()

with torch.no_grad():
    prior_sm = model_sm.forward(t_sm)
    f_sm = prior_sm.sample()

NOISE_SM = 0.10
y_sm = f_sm + NOISE_SM * torch.randn_like(f_sm)
yerr_sm = torch.full_like(f_sm, NOISE_SM)

lc_sm = Lightcurve(t_sm, y_sm, yerr=yerr_sm, max_samples=None)
print(f"Created SMK light curve: {len(lc_sm.xdata)} observations")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(
    t_sm_np,
    f_sm.numpy(),
    "-",
    color="darkorchid",
    lw=1.2,
    alpha=0.7,
    label="GP prior sample",
)
ax.errorbar(
    t_sm_np,
    y_sm.numpy(),
    yerr=NOISE_SM,
    fmt="k.",
    elinewidth=0.6,
    alpha=0.5,
    label="Noisy observations",
)
ax.set_xlabel("Time (days)")
ax.set_ylabel("Flux")
ax.set_title(
    f"Mock light curve from Spectral Mixture GP prior\n"
    f"(P\u2081 = {P1:.0f} d, P\u2082 = {P2:.0f} d; relative weights {W1:.2f} / {W2:.2f})"
)
ax.legend()
plt.tight_layout()
plt.show()

---
## 3. Fitting the mock data back with `pgmuvi`

Because the output of the sampling step is already a `Lightcurve`, you can immediately
pass it to `lc.fit()` to verify that `pgmuvi` recovers the injected kernel parameters.

Below we demonstrate this round-trip on the quasi-periodic mock data created in
Section 2a, using a modest number of training iterations as a quick sanity check.  In
a real analysis you would increase `training_iter` (typically 2000–5000) and tune the
initial period guess.

In [ ]:
torch.manual_seed(SEED)

# Fit a quasi-periodic model to the mock data from Section 2a.
# We supply the true period as an initial guess via set_hypers.
fit_result = lc_qp.fit(
    model="1DQuasiPeriodic",
    period=TRUE_PERIOD,      # initial period guess (days)
    training_iter=500,       # increase for a production run
    miniter=500,
    lr=0.05,
)

# Inspect the fitted period
fitted_period = (
    lc_qp.model.covar_module.base_kernel.kernels[0]
    .period_length.item()
)
print(f"Injected period : {TRUE_PERIOD:.1f} days")
print(f"Recovered period: {fitted_period:.1f} days")

In [ ]:
# Plot the GP posterior mean and confidence band over the mock data
fig = lc_qp.plot(show=False)
fig.axes[0].set_title(
    f"Quasi-periodic GP fitted to mock data\n"
    f"(injected P = {TRUE_PERIOD} d, recovered P ≈ {fitted_period:.1f} d)"
)
plt.tight_layout()
plt.show()

---
## 4. Drawing posterior predictive samples after fitting real data

Once you have fitted `pgmuvi` to your *actual* observations, you can also draw samples
from the **posterior predictive distribution** — i.e. GP realisations conditioned on the
data.  These are useful for visualising the uncertainty in the model prediction and for
generating mock datasets that are consistent with what was observed.

After calling `lc.fit()`, the model is in training mode.  To predict, switch to
evaluation mode first.

In [ ]:
torch.manual_seed(SEED)

# Dense time grid for smooth posterior realisations
t_pred = torch.linspace(
    float(lc_qp.xdata.min()),
    float(lc_qp.xdata.max()),
    500,
)

lc_qp._eval()   # switch model + likelihood to eval mode

with torch.no_grad(), gpytorch.settings.fast_pred_var():
    # Transform the prediction grid to the same space that the model was
    # trained on (the Lightcurve stores the raw data without transform by
    # default; xtransform=None means no rescaling is needed here).
    if lc_qp.xtransform is None:
        t_pred_transformed = t_pred
    else:
        t_pred_transformed = lc_qp.xtransform.transform(
            t_pred.to(lc_qp.xtransform.min.device)
        )

    # Posterior predictive distribution at t_pred
    posterior = lc_qp.likelihood(lc_qp.model(t_pred_transformed))

    # Draw 5 independent posterior realisations
    n_post = 5
    post_samples = posterior.sample(torch.Size([n_post]))  # shape (n_post, 500)

    mean = posterior.mean
    lower, upper = posterior.confidence_region()

fig, ax = plt.subplots(figsize=(10, 5))

# 95 % confidence region
ax.fill_between(
    t_pred.numpy(),
    lower.numpy(),
    upper.numpy(),
    alpha=0.25,
    color="steelblue",
    label="95% confidence",
)

# Posterior mean
ax.plot(
    t_pred.numpy(),
    mean.numpy(),
    "-",
    color="steelblue",
    lw=2,
    label="Posterior mean",
)

# Posterior realisations (mock data consistent with the fit)
for i, s in enumerate(post_samples):
    label = "Posterior realisations" if i == 0 else None
    ax.plot(
        t_pred.numpy(),
        s.numpy(),
        "-",
        color="orange",
        lw=0.8,
        alpha=0.6,
        label=label,
    )

# Observed data
ax.errorbar(
    lc_qp.xdata.numpy(),
    lc_qp.ydata.numpy(),
    yerr=lc_qp.yerr.numpy(),
    fmt="k.",
    elinewidth=0.8,
    alpha=0.7,
    label="Mock observations",
    zorder=5,
)

ax.set_xlabel("Time (days)")
ax.set_ylabel("Flux")
ax.set_title("Posterior predictive realisations from the fitted GP")
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.show()

---
## Summary

| Goal | What to use |
|---|---|
| Quick sinusoidal mock data | `pgmuvi.synthetic.make_simple_sinusoid_1d()` |
| GP prior sample with quasi-periodic kernel | `QuasiPeriodicGPModel` + `model.forward(t).sample()` |
| GP prior sample with stochastic / red-noise kernel | `MaternGPModel` + `model.forward(t).sample()` |
| GP prior sample with flexible spectral-mixture kernel | `SpectralMixtureGPModel` + `model.forward(t).sample()` |
| Posterior predictive samples after fitting data | `lc.likelihood(lc.model(t)).sample()` |

In all GP-prior cases the workflow is identical:

1. Construct a `pgmuvi` GP model with placeholder training data.
2. Set the desired kernel hyperparameters.
3. Call `model.eval()`, then `model.forward(t)` inside `torch.no_grad()`.
4. Draw samples with `.sample()` (or `.sample(torch.Size([n]))` for multiple realisations).
5. Add observational noise and wrap in `Lightcurve` for further analysis.